<a href="https://colab.research.google.com/github/swirita/salmonellosis-forecasting-analysis/blob/main/notebooks/03_salmonellosis_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Salmonellosis Forecasting

## Imports

In [25]:
%pip install -q pmdarima

In [26]:
# imports
import os
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
import statsmodels.tsa.api as tsa
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error

## Custom Functions

In [28]:
def plot_forecast(ts_train, ts_test, forecast_df, n_train_lags=None,
                  figsize=(10,4), title='Comparing Forecast vs. True Data'):
    ### PLot training data, and forecast (with upper/,lower ci)
    fig, ax = plt.subplots(figsize=figsize)
    # setting the number of train lags to plot if not specified
    if n_train_lags==None:
        n_train_lags = len(ts_train)

    # Plotting Training  and test data
    ts_train.iloc[-n_train_lags:].plot(ax=ax, label="train")
    ts_test.plot(label="test", ax=ax)
    # Plot forecast
    forecast_df['mean'].plot(ax=ax, color='green', label="forecast")
    # Add the shaded confidence interval
    ax.fill_between(forecast_df.index,
                    forecast_df['mean_ci_lower'],
                   forecast_df['mean_ci_upper'],
                   color='green', alpha=0.3,  lw=2)
    # set the title and add legend
    ax.set_title(title)
    ax.legend();

    return fig, ax

In [29]:
# Custom function for Ad Fuller Test
def get_adfuller_results(ts, alpha=.05, label='adfuller', **kwargs): #kwargs for adfuller()
    # Saving each output
    (test_stat, pval, nlags, nobs, crit_vals_d,
    icbest ) = tsa.adfuller(ts, **kwargs)
    # Converting output to a dictionary with the interpretation of p
    adfuller_results = {'Test Statistic': test_stat,
                        "# of Lags Used":nlags,
                       '# of Observations':nobs,
                        'p-value': round(pval,6),
                        'alpha': alpha,
                       'sig/stationary?': pval < alpha}
    return pd.DataFrame(adfuller_results, index =[label])

In [30]:
### NEW FUNCTION FOR COMBINED ACF/PACF WITH ANNOTATIONS
def plot_acf_pacf(ts, nlags=40, figsize=(10, 5),
                  annotate_sig=False, alpha=.05,
                 acf_kws={}, pacf_kws={},
                  annotate_seas=False, m = None,
                 seas_color='black'):

    fig, axes = plt.subplots(nrows=2, figsize=figsize)

    # Sig lags line style
    sig_vline_kwargs = dict( ls=':', lw=1, zorder=0, color='red')
    # ACF
    tsa.graphics.plot_acf(ts, ax=axes[0], lags=nlags, **acf_kws)

    ## Annotating sig acf lags
    if annotate_sig == True:
        sig_acf_lags = get_sig_lags(ts,nlags=nlags,alpha=alpha, type='ACF')
        for lag in sig_acf_lags:
            axes[0].axvline(lag,label='sig', **sig_vline_kwargs )
    # PACF
    tsa.graphics.plot_pacf(ts,ax=axes[1], lags=nlags, **pacf_kws)

    ## Annotating sig pacf lags
    if annotate_sig == True:
        ## ANNOTATING SIG LAGS
        sig_pacf_lags = get_sig_lags(ts,nlags=nlags,alpha=alpha, type='PACF')
        for lag in sig_pacf_lags:
            axes[1].axvline(lag, label='sig', **sig_vline_kwargs)

    ### ANNOTATE SEASONS
    if annotate_seas == True:
        # Ensure m was defined
        if m is None:
            raise Exception("Must define value of m if annotate_seas=True.")
        ## Calculate number of complete seasons to annotate
        n_seasons = nlags//m
        # Seasonal Lines style
        seas_vline_kwargs = dict( ls='--',lw=1, alpha=.7, color=seas_color, zorder=-1)

        ## for each season, add a line
        for i in range(1, n_seasons+1):
            axes[0].axvline(m*i, **seas_vline_kwargs, label="season")
            axes[1].axvline(m*i, **seas_vline_kwargs, label="season")
    fig.tight_layout()

    return fig


In [31]:
def regression_metrics_ts(ts_true, ts_pred, label="", verbose=True, output_dict=False,):
    # Get metrics
    mae = mean_absolute_error(ts_true, ts_pred)
    mse = mean_squared_error(ts_true, ts_pred)
    rmse = root_mean_squared_error(ts_true, ts_pred)
    r_squared = r2_score(ts_true, ts_pred)
    mae_perc = mean_absolute_percentage_error(ts_true, ts_pred) * 100

    if verbose == True:
        # Print Result with label
        header = "---" * 20
        print(header, f"Regression Metrics: {label}", header, sep="\n")
        print(f"- MAE = {mae:,.3f}")
        print(f"- MSE = {mse:,.3f}")
        print(f"- RMSE = {rmse:,.3f}")
        print(f"- R^2 = {r_squared:,.3f}")
        print(f"- MAPE = {mae_perc:,.2f}%")

    if output_dict == True:
        metrics = {
            "Label": label,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R^2": r_squared,
            "MAPE(%)": mae_perc,
        }
        return metrics

## Load and Prepare Data

In [32]:
# Connect Google Colab to Google Drive
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
# Project folders
project_folder = "/content/drive/MyDrive/Salmonellosis Forecasting and Analysis"
data_folder = os.path.join(project_folder, "data")

# Load the cleaned data created in the preparation notebook
cleaned_file_path = os.path.join(data_folder, "salmonellosis_weekly_cleaned.csv")
df = pd.read_csv(cleaned_file_path)

print(f"Data loaded: {df.shape[0]:,} rows and {df.shape[1]} columns")
display(df.head())


Data loaded: 17,079 rows and 10 columns


,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,Previous 52 week Max,Cumulative YTD Current MMWR Year,Cumulative YTD Previous MMWR Year,Season,Previous week cases
0,ALABAMA,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,10,38.0,10,6.0,Winter,2.0
1,ALABAMA,2022,2,Salmonellosis (excluding Salmonella Typhi infe...,3,38.0,18,12.0,Winter,10.0
2,ALABAMA,2022,3,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,24,17.0,Winter,3.0
3,ALABAMA,2022,4,Salmonellosis (excluding Salmonella Typhi infe...,2,38.0,26,25.0,Winter,0.0
4,ALABAMA,2022,5,Salmonellosis (excluding Salmonella Typhi infe...,0,38.0,29,30.0,Winter,2.0


In [34]:
# Select the national weekly totals
national = df[
    df["Reporting Area"] == "US RESIDENTS"
].copy()

# Arrange the observations correctly
national = national.sort_values(
    ["Current MMWR Year", "MMWR WEEK"]
)

# Create one weekly date for every observation
national["Date"] = pd.date_range(
    start="2022-01-08",
    periods=len(national),
    freq="W-SAT"
)

# Use Date as the index
national = national.set_index("Date")

national.head()

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,Previous 52 week Max,Cumulative YTD Current MMWR Year,Cumulative YTD Previous MMWR Year,Season,Previous week cases
Date,,,,,,,,,,
2022-01-08,US RESIDENTS,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,96,1212.0,96,333.0,Winter,2.0
2022-01-15,US RESIDENTS,2022,2,Salmonellosis (excluding Salmonella Typhi infe...,77,1217.0,415,677.0,Winter,96.0
2022-01-22,US RESIDENTS,2022,3,Salmonellosis (excluding Salmonella Typhi infe...,24,1217.0,535,1016.0,Winter,77.0
2022-01-29,US RESIDENTS,2022,4,Salmonellosis (excluding Salmonella Typhi infe...,46,1217.0,663,1387.0,Winter,24.0
2022-02-05,US RESIDENTS,2022,5,Salmonellosis (excluding Salmonella Typhi infe...,65,1227.0,977,1699.0,Winter,46.0


In [35]:
# Keep only the column we want to forecast
weekly_cases = national[["Current week"]].copy()

# Give it a clearer name
weekly_cases = weekly_cases.rename(
    columns={"Current week": "Cases"}
)

# Make sure the values are numeric
weekly_cases["Cases"] = pd.to_numeric(
    weekly_cases["Cases"],
    errors="coerce"
)

weekly_cases.head()

,Cases
Date,
2022-01-08,96
2022-01-15,77
2022-01-22,24
2022-01-29,46
2022-02-05,65


In [36]:
print("First date:", weekly_cases.index.min())
print("Last date:", weekly_cases.index.max())
print("Number of weeks:", len(weekly_cases))
print("Missing case values:", weekly_cases["Cases"].isna().sum())

First date: 2022-01-08 00:00:00
Last date: 2026-09-05 00:00:00
Number of weeks: 244
Missing case values: 0


In [37]:
weekly_cases.index

DatetimeIndex(['2022-01-08', '2022-01-15', '2022-01-22', '2022-01-29',
               '2022-02-05', '2022-02-12', '2022-02-19', '2022-02-26',
               '2022-03-05', '2022-03-12',
               ...
               '2026-07-04', '2026-07-11', '2026-07-18', '2026-07-25',
               '2026-08-01', '2026-08-08', '2026-08-15', '2026-08-22',
               '2026-08-29', '2026-09-05'],
              dtype='datetime64[ns]', name='Date', length=244, freq=None)

In [38]:
pd.infer_freq(weekly_cases.index)

'W-SAT'

In [39]:
weekly_cases = weekly_cases.asfreq("W-SAT")
weekly_cases.isna().sum()

,0
Cases,0


In [40]:
weekly_cases.index

DatetimeIndex(['2022-01-08', '2022-01-15', '2022-01-22', '2022-01-29',
               '2022-02-05', '2022-02-12', '2022-02-19', '2022-02-26',
               '2022-03-05', '2022-03-12',
               ...
               '2026-07-04', '2026-07-11', '2026-07-18', '2026-07-25',
               '2026-08-01', '2026-08-08', '2026-08-15', '2026-08-22',
               '2026-08-29', '2026-09-05'],
              dtype='datetime64[ns]', name='Date', length=244, freq='W-SAT')

* note: during analysis , the data had a really high seasonality pattern in warm months of the summer , so a seasonal model is requeired. (m=52) which is equal to 12 months.

**REMEMBER THE DATA IS WEEKLY NOT MONTHLY SO WE USE 52.**

`SEASON_LENGTH= 52`
`TEST_SIZE= 52`
`FORECASTING_STEPS= 4`



## Forecast

### 1. Differencing, Stationarity, ACF, and PACF

### 2. Train/Test Split and Manual SARIMA

### 3. Tune wuth `auto_arima`

### 4. Select the Final Model and Forecast the Future

## Final Evaluation